In [1]:
spark.conf.set("spark.sql.shuffle.partitions", "50")
spark.conf.set("spark.default.parallelism", "50")

train = train.repartition(50)
test = test.repartition(50)

train.cache()
test.cache()
print("Cached")


NameError: name 'spark' is not defined

In [2]:
from spark_start import spark
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)

print("Spark ready")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/16 00:50:40 WARN Utils: Your hostname, Saideep-Reddys-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.2 instead (on interface en0)
26/02/16 00:50:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 00:50:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/16 00:50:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/16 00:50:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/16 00:50:42 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark started successfully
Spark ready


In [3]:
df = spark.read.parquet(f"{PROJECT_ROOT}/data/processed/final")

train, test = df.randomSplit([0.8, 0.2], seed=42)

print("Train:", train.count())
print("Test:", test.count())


Train: 2032330


[Stage 4:==============>                                            (2 + 6) / 8]

Test: 507717


In [4]:
spark.conf.set("spark.sql.shuffle.partitions", "50")
spark.conf.set("spark.default.parallelism", "50")

train = train.repartition(50)
test = test.repartition(50)

train.cache()
test.cache()
print("Cached")


Cached


In [5]:
spark.conf.set("spark.sql.shuffle.partitions", "50")
spark.conf.set("spark.default.parallelism", "50")

train = train.repartition(50)
test = test.repartition(50)

train.cache()
test.cache()
print("Cached")


Cached


In [6]:
spark.conf.set("spark.sql.shuffle.partitions", "50")
spark.conf.set("spark.default.parallelism", "50")

train = train.repartition(50)
test = test.repartition(50)

train.cache()
test.cache()

train.count()
test.count()
print("Cached successfully")


[Stage 45:===========================================>            (39 + 8) / 50]

Cached successfully


In [7]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label_binary",
    maxIter=20
)

lr_model = lr.fit(train)
lr_pred = lr_model.transform(test)

lr_metrics = evaluate(lr_pred)
print("LR:", lr_metrics)


                                                                                26/02/16 00:53:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

NameError: name 'evaluate' is not defined

In [8]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate(predictions):

    acc = MulticlassClassificationEvaluator(
        labelCol="label_binary",
        metricName="accuracy"
    ).evaluate(predictions)

    f1 = MulticlassClassificationEvaluator(
        labelCol="label_binary",
        metricName="f1"
    ).evaluate(predictions)

    precision = MulticlassClassificationEvaluator(
        labelCol="label_binary",
        metricName="weightedPrecision"
    ).evaluate(predictions)

    recall = MulticlassClassificationEvaluator(
        labelCol="label_binary",
        metricName="weightedRecall"
    ).evaluate(predictions)

    auc = BinaryClassificationEvaluator(
        labelCol="label_binary"
    ).evaluate(predictions)

    return acc, f1, precision, recall, auc


In [9]:
lr_metrics = evaluate(lr_pred)
print("LR:", lr_metrics)


[Stage 229:====================================>                  (33 + 8) / 50]

LR: (0.9866047424057103, 0.9867157925807732, 0.9869316706861704, 0.9866047424057103, 0.9978881176147256)


In [10]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    labelCol="label_binary",
    featuresCol="features",
    maxDepth=8
)

dt_model = dt.fit(train)
dt_pred = dt_model.transform(test)

dt_metrics = evaluate(dt_pred)
print("DT:", dt_metrics)


DT: (0.9897856483040749, 0.9897078163879423, 0.9897171832903359, 0.9897856483040749, 0.9985075883138326)


In [11]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label_binary",
    featuresCol="features",
    numTrees=25,
    maxDepth=8,
    subsamplingRate=0.7
)

rf_model = rf.fit(train)
rf_pred = rf_model.transform(test)

rf_metrics = evaluate(rf_pred)
print("RF:", rf_metrics)


RF: (0.9923441602309948, 0.9923371099747499, 0.99233135913622, 0.9923441602309948, 0.9995904234934186)


In [12]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    labelCol="label_binary",
    featuresCol="features",
    maxIter=15,
    maxDepth=6
)

gbt_model = gbt.fit(train)
gbt_pred = gbt_model.transform(test)

gbt_metrics = evaluate(gbt_pred)
print("GBT:", gbt_metrics)


GBT: (0.9924150658733113, 0.9923938486392253, 0.9923845876127622, 0.9924150658733113, 0.9996448909335872)


In [13]:
import pandas as pd

results = pd.DataFrame({
    "Model":["Logistic Regression","Decision Tree","Random Forest","Gradient Boosted Trees"],
    "Accuracy":[lr_metrics[0],dt_metrics[0],rf_metrics[0],gbt_metrics[0]],
    "F1":[lr_metrics[1],dt_metrics[1],rf_metrics[1],gbt_metrics[1]],
    "Precision":[lr_metrics[2],dt_metrics[2],rf_metrics[2],gbt_metrics[2]],
    "Recall":[lr_metrics[3],dt_metrics[3],rf_metrics[3],gbt_metrics[3]],
    "AUC":[lr_metrics[4],dt_metrics[4],rf_metrics[4],gbt_metrics[4]]
})

results.to_csv("tableau/csv_exports/model_results.csv", index=False)
results


,Model,Accuracy,F1,Precision,Recall,AUC
0,Logistic Regression,0.986605,0.986716,0.986932,0.986605,0.997888
1,Decision Tree,0.989786,0.989708,0.989717,0.989786,0.998508
2,Random Forest,0.992344,0.992337,0.992331,0.992344,0.999590
3,Gradient Boosted Trees,0.992415,0.992394,0.992385,0.992415,0.999645
